In [2]:
import pandas as pd
import polychrom
from polychrom.hdf5_format import list_URIs, load_URI
import noctiluca as nl
import bayesmsd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import matplotlib.ticker as ticker
import os
from multiprocessing import Pool
from functools import partial
import glob
import pickle
import matplotlib
matplotlib.set_loglevel('error')
from scipy import stats
import itertools
from polychrom.contactmaps import monomerResolutionContactMap, monomerResolutionContactMapSubchains, binnedContactMap
from multiprocessing import Pool 
from polychrom import polymerutils, polymer_analyses
from cooltools.lib.numutils import observed_over_expected
from copy import deepcopy
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
# plot formatting
plt.rcParams["font.size"] = 6

plt.rcParams.update({
    "text.usetex": False})

# makes illustrator recognize text as text objects
plt.rc('pdf',fonttype = 42) 
from matplotlib.ticker import EngFormatter
bp_formatter = EngFormatter('b')
def format_ticks(ax, x=True, y=True, rotate=True):
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
    if rotate:
        ax.tick_params(axis='y',rotation=45)
        ax.tick_params(axis='x',rotation=45)
        for tick in ax.get_yticklabels():
            tick.set_verticalalignment('center')

# sns.set_style('whitegrid')

# plot formatting (Hansen Lab style guide)
plt.rc("pdf", fonttype=42)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["font.size"] = 6
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.linewidth"] = 0.25
plt.rcParams["axes.titlesize"] = 6
plt.rcParams["axes.labelsize"] = 6
plt.rcParams["xtick.major.width"] = 0.25
plt.rcParams["xtick.labelsize"] = 6
plt.rcParams["ytick.major.width"] = 0.25
plt.rcParams["ytick.labelsize"] = 6
plt.rcParams["legend.fontsize"] = 6
plt.rcParams["legend.frameon"] = False
plt.rcParams["legend.facecolor"] = "none"
plt.rcParams["lines.linewidth"] = 1.00
plt.rcParams["text.usetex"] = False
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["ps.useafm"] = True

# colors (Hansen Lab style guide)
colors = {"blue": (0.000,0.467,0.733),
          "cyan": (0.200,0.733,0.933),
          "teal": (0.000,0.600,0.533),
          "orange": (0.933,0.467,0.200),
          "red": (0.800,0.200,0.067),
          "magenta": (0.933,0.200,0.467),
          "grey": (0.733,0.733,0.733)}

### Contact maps

In [ ]:
# utility functions
def insulation(map,windows,normalize=True):
	insus = []
	matlen=map.shape[0]
	for ws in windows:
		if matlen<ws*2:
			insus.append(np.zeros(matlen))
		else:
			ins = np.full(matlen,np.nan)
			for row in range(ws, matlen - ws):
				sub_mat = map[row - ws: row, row + 1: row + ws + 1]
				# print(sub_mat)
				ins[row] = np.nanmean(sub_mat)

			if normalize:
				ins = np.log2(ins / np.nanmean(ins))
				ins[~np.isfinite(ins)] = 0
			insus.append(ins)
	return insus[0] if len(insus)==1 else insus

def sinkhorn_knopp(mat, tol=1e-11, max_iter=1000, verbose=False):
	out_mat = mat.astype(np.float64)
	total_iterations = 0
	for i in range(max_iter):
		# do row and column normalization
		out_mat /= np.sum(out_mat,axis=1,keepdims=True)
		out_mat /= np.sum(out_mat,axis=0,keepdims=True)
		# check for convergence
		rowsums = np.sum(out_mat,axis=1)
		colsums = np.sum(out_mat,axis=0)
		total_iterations+=1
		if np.allclose(rowsums,1,rtol=0,atol=tol) and np.allclose(colsums,1,rtol=0,atol=tol):
			if verbose:
				print(f"Convergence reached after {total_iterations} iteration(s)")
			return out_mat
	if verbose:
		print("Max iterations reached without convergence")
	return out_mat

def get_contact_map(datadir,starts,mapN,balance=True):
	URIs = list_URIs(datadir)
	contact_map = monomerResolutionContactMapSubchains(filenames=URIs,mapStarts=starts,mapN=mapN,
													cutoff=5, n=8, loadFunction=lambda x:load_URI(x)["pos"])
	if balance:
		contact_map = sinkhorn_knopp(contact_map)
	return contact_map

In [ ]:
# Load data
datadir = './outputs' # change as needed
conds = glob.glob('*/',root_dir=datadir)
dirs = [(c[:-1],datadir+c[:-1]) for c in conds]
files = dict(dirs)

In [ ]:
# compute contact maps
maps = {}
starts = list(range(0,40080,2004)) # change as needed to match total c'some/individual repeat sizes
for condname, datadir in files.items():
    maps[condname] = get_contact_map(datadir,starts,2004)
with open('maps.pkl','rb') as file:
    pickle.dump(maps,file) # pickle contact maps for later use

In [ ]:
# plot raw contact maps (Fig. 5a,b, S5)
def plot_contact_map(cond,map,savepath=None):
	fig,ax=plt.subplots(figsize=(1.75,1.75))
	im=ax.matshow(np.log(map),cmap='YlOrBr')
	divider = make_axes_locatable(ax)
	cax = divider.append_axes('right',size='5%',pad=0.02)
	plt.colorbar(im,cax=cax,label='log(counts)')
	ax.xaxis.tick_bottom()
	ax.set_xticklabels([])
	if savepath is not None:
	    plt.savefig(f'{savepath}/{cond}_raw.svg',
	            transparent=True,bbox_inches='tight')
savepath='./figures' # change as desired
for cond,map in maps.items():
	plot_contact_map(cond,map,savepath=savepath)

In [ ]:
# plot differential contact maps with differential insulation profiles
# (Fig. 4b-e, 5a,b)
def plot_diff_panel(hmap,base_map,savepath=None,vline=False,
                    hide_x=False,hide_y=False,
                    vmax=None,ins_vmax=None):
    diff_map = np.log(hmap) - np.log(base_map)
    diff_ins = insulation(base_map,[100])-insulation(hmap,[100])
    if vmax is None:
        vmax = np.max([np.abs(np.nanmin(diff_map)),np.abs(np.nanmax(diff_map))])
    fig,ax = plt.subplots(figsize=(1.75,1.75))
    im = ax.matshow(diff_map,cmap='bwr',vmax=vmax,vmin=-vmax)
    ax.tick_params(axis='y',labelrotation=45)
    # im = ax.matshow(diff_map,cmap='bwr')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right',size='5%',pad=0.02)
    plt.colorbar(im,cax=cax)
    plt.yticks(rotation=45,va='bottom')
    ax1 = divider.append_axes('bottom',size='40%',pad=0.1)
    ax1.plot(np.arange(0,2004,1),diff_ins,color='#ee3377')
    ax1.plot(np.arange(0,2004,1),[0]*2004,'k--',alpha=0.5)
    ax.set_xticks([])
    ax1.set_xlim(0,2004)
    ax1.set_ylim(top=ins_vmax)
    ax1.tick_params(axis='y',labelrotation=45)
    ax.spines[['top','right']].set_visible(True)
    if vline:
        # ax.axvline(x=622,ymin=-0.2,ymax=1,c='k',clip_on=False,ls=':',alpha=0.5)
        ax1.axvline(x=622,ymin=0,ymax=1.3,c='k',clip_on=False,ls='--',alpha=0.5)
    if hide_x:
        ax1.set_xticklabels([])
    if hide_y:
        ax.set_yticklabels([])
    if savepath is not None:
        plt.savefig(savepath,transparent=True,bbox_inches='tight',dpi=300)

for cond, hmap in maps.items():
    if "baseline" in cond:
        continue
    if "strong" in cond or "4" in cond:
        hide_x=False
    else:
        hide_x=True
    savepath = './figures/'+cond+'.svg' # change as desired
    if "decomp" in cond:
        strength = cond.split('_')[-1]
        plot_diff_panel(hmap,maps[f'baseline_sticky_{strength}'],savepath=savepath,vline=True,
                        hide_x=hide_x,hide_y=False)
    else:
        plot_diff_panel(hmap,maps['baseline'],savepath=savepath,vline=True,
                        hide_x=hide_x,hide_y=True)


### Looped probability and cohesin position analysis

In [ ]:
# utility functions
REGION_SIZE=40080
def _connected(SMC_pos):
    """Transitive closure of "these two LEFs share a leg (within +/-1 site)".

    Same relation as in_contact() in analysis.ipynb, but built once per snapshot
    instead of once per anchor pair.
    """
    ext = np.concatenate((SMC_pos, SMC_pos + 1, SMC_pos - 1), axis=1)  # (M, 6)
    adj = np.any(ext[:, None, :, None] == ext[None, :, None, :], axis=(2, 3)).astype(np.int32)
    while True:
        new = ((adj + adj @ adj) > 0).astype(np.int32)
        if np.array_equal(new, adj):
            return new.astype(bool)
        adj = new


def loop_probs(data, pairs, chrom_size, region_size=REGION_SIZE,
               block_start=0, block_end=None, block_step=10):
    """Looping probability of each anchor pair, averaged over blocks then regions.

    data  (numBlocks, Nlefs, 2) array of LEF leg positions
    returns {pair: probability}, plus the number of blocks and regions averaged over.
    """
    if block_end is None:
        block_end = data.shape[0]
    block_end = min(block_end, data.shape[0])
    blocks = range(block_start, block_end, block_step)
    num_regions = chrom_size // region_size
    region_starts = np.arange(num_regions) * region_size
    anchors = sorted({a for pair in pairs for a in pair})

    # contacts[pair] counts, per region, the snapshots where the two anchors are linked
    contacts = {pair: np.zeros(num_regions) for pair in pairs}
    for region_num, region_start in enumerate(region_starts):
        region_end = region_start + region_size
        counts = {pair: 0 for pair in pairs}
        n_blocks = 0
        for block_num in blocks:
            n_blocks += 1
            SMC_pos_all = data[block_num, :, :]
            # keep only LEFs with both legs inside this region, and offset to [0, region_size)
            keep = np.all(SMC_pos_all >= region_start, 1) & np.all(SMC_pos_all < region_end, 1)
            SMC_pos = SMC_pos_all[keep, :] - region_start
            if SMC_pos.shape[0] == 0:
                continue
            idx = {}
            for a in anchors:
                hits = np.where(SMC_pos == a)[0]
                if hits.size:
                    idx[a] = hits[0]
            todo = [p for p in pairs if p[0] in idx and p[1] in idx]
            if not todo:
                continue
            linked = _connected(SMC_pos)
            for pair in todo:
                if linked[idx[pair[0]], idx[pair[1]]]:
                    counts[pair] += 1
        for pair in pairs:
            contacts[pair][region_num] = counts[pair] / n_blocks if n_blocks else np.nan

    probs = {pair: float(np.mean(contacts[pair])) for pair in pairs}
    return probs, len(blocks), num_regions

def calculate_GT_loop_prob(datadir,chrom_size=40080,repeat_size=2004,
                                       left=155,right=1166,start=219,end=622):
    lefnums = sorted([x[3:-4] for x in glob.glob('SMC*.dat',root_dir=datadir)],
                                  key=int)
    lefpaths = [datadir+"/SMC"+x+".dat" for x in lefnums]
    LEFs = []
    # if len(lefpaths)==0:
    #     return None
    for f in lefpaths:
        with open(f,'rb') as file:
            lef = pickle.load(file)
        lef = [[*legs] for legs in lef]
        LEFs.append(lef)
    LEFs = np.array(LEFs)
    n_repeats = chrom_size//repeat_size
    loop_fracs = []
    for i in range(n_repeats):
        region_left = left+i*repeat_size
        region_right = right+i*repeat_size
        region_start = start+i*repeat_size
        region_end = end+i*repeat_size
        lf,_,_ = loop_probs(LEFs,[(region_left,region_right)],
                                chrom_size,region_size=chrom_size,
                                block_step=1)
        if np.isnan(lf[(region_left,region_right)]):
            lf[(region_left,region_right)]=0
        loop_fracs.append(lf[(region_left,region_right)])
    return loop_fracs
    
def calculate_GT_loop_prob_by_PolII(datadir,chrom_size=40080,repeat_size=2004,
                                       left=155,right=1166,start=219,end=622):
    lefnums = sorted([x[3:-4] for x in glob.glob('SMC*.dat',root_dir=datadir)],
                              key=int)
    lefpaths = [datadir+"/SMC"+x+".dat" for x in lefnums]
    LEFs = []
    if len(lefpaths)==0:
        return None
    for f in lefpaths:
        with open(f,'rb') as file:
            lef = pickle.load(file)
        lef = [[*legs] for legs in lef]
        LEFs.append(lef)
    LEFs = np.array(LEFs)
    print(LEFs.shape)
    polnums = sorted([x[4:-4] for x in glob.glob('pols*.dat',root_dir=datadir)],
                              key=int)
    polpaths = [datadir+"/pols"+x+".dat" for x in polnums]
    pols = []
    if len(polpaths)==0:
        return None
    for f in polpaths:
        with open(f,'rb') as file:
            p = pickle.load(file)
        pols.append(p.flatten())
    n_repeats = chrom_size//repeat_size
    total_loop_fracs = []
    on_loop_fracs = []
    off_loop_fracs = []
    for i in range(n_repeats):
        region_left = left+i*repeat_size
        region_right = right+i*repeat_size
        region_start = start+i*repeat_size
        region_end = end+i*repeat_size
        burst = np.array([np.any((p>=region_start)&(p<=region_end)) for p in pols])
        on_LEFs = LEFs[burst,:,:]
        off_LEFs = LEFs[~burst,:,:]
        # print((len(on_LEFs),len(off_LEFs)))
        on_loopf, _, _ = loop_probs(on_LEFs,[(region_left,region_right)],
                                        chrom_size,region_size=chrom_size,
                                        block_step=1)
        on_loop_fracs.append(on_loopf[(region_left,region_right)])
        off_loopf, _, _ = loop_probs(off_LEFs,[(region_left,region_right)],
                                        chrom_size,region_size=chrom_size,
                                        block_step=1)
        off_loop_fracs.append(off_loopf[(region_left,region_right)])
        total_lf,_,_ = loop_probs(LEFs,[(region_left,region_right)],
                                        chrom_size,region_size=chrom_size,
                                        block_step=1)
        total_loop_fracs.append(total_lf[(region_left,region_right)])
    return {'overall':total_loop_fracs,'RNAPII ON':on_loop_fracs,'RNAPII OFF':off_loop_fracs}


def sim_SMC_ChIP(input, region_size=2004):
    condname = input[0]
    files = glob.glob(f'{input[1]}/SMC*.dat')
    SMCs = []
    for block in files:
        with open(block,'rb') as f:
            s = pickle.load(f)
        SMCs = SMCs + s
    SMCs = np.array(SMCs)
    SMCs = SMCs%region_size
    return condname, SMCs

In [ ]:
# calculate ground truth looped probabilities from LEF positions
overall_loop_fracs = {}
for cond,datadir in files.items():
    overall_loop_fracs[cond]=calculate_GT_loop_prob(datadir)
looped_fracs_by_PolII = {}
for cond,datadir in files.items():
	looped_fracs_by_PolII[cond] = calculate_GT_loop_prob_by_PolII(datadir)

In [ ]:
# plot looped probabilities
conds_to_plot = ['cond1']
lfs_to_plot = dict([(c,overall_loop_fracs[c]) for c in conds_to_plot])
fig,ax = plt.subplots(figsize=(1.3,1.5))
colors = {"cond1":'#bbbbbb'}
sns.boxplot(lfs_to_plot,order=conds_to_plot,ax=ax,palette=colors,fliersize=1)
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_visible(True)
ax.spines['left'].set_color('black')
ax.spines['left'].set_linewidth(0.5)

ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_color('black')
ax.spines['bottom'].set_linewidth(0.5)
ax.set_ylabel('Looped fraction')
ax.set_xticklabels(conds_to_plot,
                   rotation=45,ha='right')
plt.savefig('./looped_fracs_overall.svg',
            transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
# plot looped probabilities by PolII
lfs_to_plot = looped_fracs_by_PolII['cond1']
fig,ax = plt.subplots(figsize=(1,1.5))
colors = {'overall':'#0077bb',
          'RNAPII ON':'#009988',
          'RNAPII OFF':'#bbbbbb'}
sns.boxplot(lfs_to_plot,order=['overall','RNAPII ON','RNAPII OFF'],ax=ax,palette=colors,
            fliersize=1)
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_visible(True)
ax.spines['left'].set_color('black')
ax.spines['left'].set_linewidth(0.5)

ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_color('black')
ax.spines['bottom'].set_linewidth(0.5)
ax.set_ylabel('Looped fraction')
ax.set_xticklabels(['overall','RNAPII ON','RNAPII OFF'],
                   rotation=45,ha='right')
plt.savefig('./looped_fracs_byRNAPII.svg',
            transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
# perform simulated cohesin ChIP-seq by profiling LEF positions
with Pool() as pool:
    chip_list = pool.map(sim_SMC_ChIP,files.items())
    chip_dict = dict(chip_list)

In [ ]:
# plot simulated cohesin ChIP-seq
fig,ax = plt.subplots(figsize=(1.5,1.2))
ax.hist(chip_dict['cond1'].flatten(),100,histtype='step',color='k')
ax.hist(chip_dict['cond2'].flatten(),100,
        histtype='step',color='#0077bb',alpha=0.7)
ax.set_xlim(0,2004)
ax.legend([plt.Line2D([0],[0],color='k',linewidth=1,alpha=0.7),
           plt.Line2D([0],[0],color='b',linewidth=1,alpha=0.7)],
           ['cond1','cond2'],
           loc='upper right',handlelength=1.5,labelspacing=0.8)
ax.set_xlabel('Position [monomers]')
ax.set_ylabel('Count')
plt.savefig('./figures/moving_barrier_SMC_ChIP.svg',
            transparent=True,bbox_inches='tight',dpi=300)

### Mean 3D distance and contact probability heatmaps (Fig. S6-9)

In [ ]:
# utility functions
def extract_pos_set(datadir, total_length, chain_length,
                    positions):
    files = list_URIs(datadir)
    n_repeats = int(total_length/chain_length)
    pos_matrix = np.zeros((len(positions),n_repeats,len(files),3))
    for i, f in enumerate(files):
        data = load_URI(f)['pos']
        for r in range(n_repeats):
            for j, p in enumerate(positions):
                p_current = p+chain_length*r
                pos_matrix[j,r,i,:] = data[p_current]
    return pos_matrix

def mean_distance_matrix(pos_mat):
    mean_dists = np.zeros((pos_mat.shape[0],pos_mat.shape[0]))
    for i in range(pos_mat.shape[0]):
        for j in range(i+1,pos_mat.shape[0]):
            dist = np.nanmean(np.linalg.norm((pos_mat[j,:,:,:]-pos_mat[i,:,:,:])*15.1,axis=2))
            mean_dists[i,j] = dist
            mean_dists[j,i] = dist
    return mean_dists

def contact_freq_matrix(pos_mat, contact_radius,percent=False):
    contacts = np.ones((pos_mat.shape[0],pos_mat.shape[0]))
    for i in range(pos_mat.shape[0]):
        for j in range(i+1,pos_mat.shape[0]):
            dist = np.linalg.norm((pos_mat[j,:,:,:]-pos_mat[i,:,:,:])*15.1,axis=2)
            cf = np.mean(np.where(dist<=contact_radius,1,0))
            contacts[i,j] = cf
            contacts[j,i] = cf
    if percent:
        contacts*=100
    return contacts

def compute_mean_dists(input, positions, total_length=40080, chain_length=2004):
    condname = input[0]
    datadir = input[1]
    pos_mat = extract_pos_set(datadir,total_length=total_length,
                              chain_length=chain_length,
                              positions=positions)
    mean_mat = mean_distance_matrix(pos_mat)
    return condname, mean_mat

def compute_contact_freq(input, positions, contact_radius,
                         total_length=40080, chain_length=2004,
                         percent=True):
    condname = input[0]
    datadir = input[1]
    pos_mat = extract_pos_set(datadir,total_length=total_length,
                              chain_length=chain_length,
                              positions=positions)
    cf_mat = contact_freq_matrix(pos_mat,contact_radius,percent=percent)
    return condname, cf_mat

In [ ]:
# compute mean 3d distances and contact probabilities (sub-50nm)
positions = [155,219,622,1166,1400,1519,1561,1874] # change as desired
#L1, Fbn2 TTS, Fbn2 TSS, R1, downstream CTCF, Isoc1 TSS, Isoc1 TTS, Adamts19 TSS
mean_func = partial(compute_mean_dists,positions=positions,total_length=40080,
                    chain_length=2004)
cf_func = partial(compute_contact_freq,positions=positions,
                  contact_radius=50,total_length=40080,
                  chain_length=2004)
with Pool() as pool:
    mean_list = pool.map(mean_func,files.items())
    mean_dist_dict = dict(mean_list)
    cf_list = pool.map(cf_func,files.items())
    cf_dict = dict(cf_list)

In [ ]:
# function to plot heatmap rows
def hmap_rowplot(dists,conds,names,pos_labels,cbar_label,cmap='YlOrBr_r',
                 fmt='.0f',vmin=0,vmax=700,savepath=None,transparent=True):
    fig,axs = plt.subplots(1,4,figsize=(7.2,1.75))
    divider = make_axes_locatable(axs[3])
    cax = divider.append_axes('right',size='5%',pad=0.02)
    for i, ax in enumerate(axs):
        if i<len(axs)-1:
            sns.heatmap(np.tril(dists[conds[i]],k=-1),cmap=cmap,
                    fmt=fmt,annot=True,ax=ax,cbar=False,
                    mask=np.triu(np.ones_like(dists[conds[i]])),
                    vmin=vmin,vmax=vmax)
        else:
            sns.heatmap(np.tril(dists[conds[i]]),cmap=cmap,
                    fmt=fmt,annot=True,ax=ax,cbar_ax=cax,
                    cbar_kws={'label':cbar_label},
                    mask=np.triu(np.ones_like(dists[conds[i]])),
                    vmin=vmin,vmax=vmax)
        ax.set_xticklabels(pos_labels,rotation=45,ha='right',va='top')
        if i>0:
            ax.set_yticks([])
        ax.set_title(names[i])
    axs[0].set_yticklabels(pos_labels,rotation=45,va='top',ha='right')
    if savepath is not None:
        plt.savefig(savepath,transparent=transparent,dpi=300,bbox_inches='tight')

# example usage to plot heatmaps of mean distances and sub-50nm probabilities
conds_to_plot = ['baseline','moving_barrier_weak','moving_barrier_mid','moving_barrier_strong']
hmap_rowplot(mean_dist_dict,conds_to_plot,names,pos_labels,
                  vmin=180,vmax=760,cbar_label='Mean 3D distance [nm]',
                  savepath='/mnt/md0/Jamie/Simulations/Fbn2_full_locus/analysis/figures/mean_dist_matrices/moving_barrier.svg')
hmap_rowplot(mean_dist_dict,conds_to_plot,names,pos_labels,
                  vmin=180,vmax=760,cbar_label='Mean 3D distance [nm]',
                  savepath='/mnt/md0/Jamie/Simulations/Fbn2_full_locus/analysis/figures/mean_dist_matrices/moving_barrier.png',
                  transparent=False)
hmap_rowplot(cf_dict,conds_to_plot,names,pos_labels,cmap='YlOrBr',fmt='.1f',
             cbar_label='Contact probability [%]',vmin=0,vmax=8,
             savepath='/mnt/md0/Jamie/Simulations/Fbn2_full_locus/analysis/figures/contact_percent_matrices/moving_barrier.svg')
hmap_rowplot(cf_dict,conds_to_plot,names,pos_labels,cmap='YlOrBr',fmt='.1f',
             cbar_label='Contact probability [%]',vmin=0,vmax=8,
             savepath='/mnt/md0/Jamie/Simulations/Fbn2_full_locus/analysis/figures/contact_percent_matrices/moving_barrier.png',
             transparent=False)

### Distance distribution and MSD analysis

In [ ]:
# function to extract trajectories from simulation data
def extract_nl_trajectories(input, total_length, chain_length,
                            left_pos, right_pos, steps_per_block, save_every,
                            save=True, 
                            savedir='/mnt/md0/Jamie/Simulations/Fbn2_full_locus/analysis/MSD_calibration/simulated_datasets'):
    condname = input[0]
    datadir = input[1]
    files = list_URIs(datadir)
    trajectories = []
    dataset = nl.TaggedSet()
    for i in range(int(total_length/chain_length)):
        traj = []
        lp = left_pos+chain_length*i
        rp = right_pos+chain_length*i
        for t in range(len(files)):
            traj.append(load_URI(files[t])['pos'][lp]-load_URI(files[t])['pos'][rp])
        traj = np.array(traj,dtype=np.float64)
        traj = nl.Trajectory(traj)
        traj.meta['Δt'] = steps_per_block*save_every # express time axis in terms of polymer steps
        dataset.add(traj)
    if save:
        filepath = f'{condname}_{steps_per_block}steps_{save_every}blocks.h5'
        os.makedirs(savedir,exist_ok=True)
        savepath = os.path.join(savedir,filepath)
        nl.io.write.hdf5(dataset,savepath)
    return condname, dataset

In [ ]:
# extract 3D distance trajectories
datadir = './outputs/'
conds = glob.glob('*/',root_dir=datadir)
dirs = [(c[:-1],datadir+c[:-1]) for c in conds]
files = dict(dirs)
total_length = 40080
chain_length = 2004
left_pos = 155
right_pos = 1166
steps_per_block = 450
save_every = 100
# nl_datasets = {}
mproc_func = partial(extract_nl_trajectories,total_length=total_length,
                     chain_length=chain_length,left_pos=left_pos,
                     right_pos=right_pos,steps_per_block=steps_per_block,
                     save_every=save_every,
                     savedir='./nl_datasets')
with Pool() as pool:
    dataset_list = pool.map(mproc_func,files.items())
    nl_datasets = dict(dataset_list)
datasets = {'sim1':nl_datasets}

In [ ]:
# load existing trajectories to avoid having to rerun
def load_trajectories(filedict):
    datadict = {}
    for anchors, datadir in filedict.items():
        anchor_dict = {}
        for f in sorted(glob.glob('*.h5',root_dir=datadir)):
            file_parts = f.split('_')
            cond = '_'.join(file_parts[:-2])
            steps = int(file_parts[-2][:-5])
            blocks = int(file_parts[-1][:-9])
            # print(steps,blocks)
            trajfile = os.path.join(datadir,f)
            dataset = nl.io.load.hdf5(trajfile)
            for t in dataset:
                t.meta['Δt'] = steps*blocks
            anchor_dict[cond] = dataset
        datadict[anchors] = anchor_dict
    return datadict
files_to_load = {'sim1':'./nl_datasets'}
datasets = load_trajectories(files_to_load)

In [ ]:
# compute 2-point MSDs
MSDs = {}
def MSD_multiproc(input):
    msd = nl.analysis.p2.MSD(input[1],recalculate=True)
    return input[0], msd
for anchors, dataset in datasets.items():
    with Pool() as pool:
        ind_msd_list = pool.map(MSD_multiproc,dataset.items())
    ind_msd = dict(ind_msd_list)
    MSDs[anchors] = ind_msd

In [ ]:
# compute 3D distance distributions
def calculate_distance_distribution(dataset,flatten=True):
    dist_dict = {}
    for cond, data in dataset.items():
        if flatten:
            dists = []
            for i in range(len(data)):
                rel_pos = data[i][:]
                d = np.linalg.norm(rel_pos,axis=1)*15.1
                dists.append(d)
            dists = np.concatenate(dists)
            dist_dict[cond] = np.array(dists)
        else:
            dists = np.zeros((len(data),len(data[0])))
            for i in range(len(data)):
                rel_pos = data[i][:]
                d = np.linalg.norm(rel_pos,axis=1)*15.1
                dists[i,:]=d
            dist_dict[cond] = dists
    return dist_dict
all_dists = {}
for anchors,dataset in datasets.items():
    all_dists[anchors] = calculate_distance_distribution(dataset)

In [ ]:
# calculate distance distribution, stratified by RNAPII presence in gene body
def calculate_distance_distribution_by_PolII(dataset,folder,
                                             gene_start=219,gene_end=622,
                                             region_size=2004,
                                             flatten=True,
                                             cond_prefix=None,
                                             cond_affix=None):
    dist_dict = {}
    for c, data in dataset.items():
        if cond_prefix is not None:
            cond = cond_prefix+c
        else:
            cond=c
        if cond_affix is not None:
            cond = cond+cond_affix
        print(cond)
        pols = []
        polnums = sorted([x[4:-4] for x in glob.glob('pols*.dat',root_dir=os.path.join(folder,cond))],
                          key=int)
        polfiles = ["pols"+x+".dat" for x in polnums]
        print(polfiles)
        if len(polfiles)<len(data[0]):
            continue
        for p in polfiles:
            with open(os.path.join(folder,cond,p),'rb') as file:
                pols.append(pickle.load(file).flatten())
        print((len(pols),len(data[0])))
        if flatten:
            on_dists = []
            off_dists = []
            for i in range(len(data)):
                rel_pos = data[i][:]
                d = np.linalg.norm(rel_pos,axis=1)*15.1
                for j in range(len(d)):
                    if np.any((gene_start+i*region_size<=pols[j])&(pols[j]<=gene_end+i*region_size)):
                        on_dists.append(d[j])
                    else:
                        off_dists.append(d[j])
            cond_dict = {'ON':on_dists,
                         'OFF':off_dists}
            dist_dict[c]=cond_dict
    return dist_dict
dists_by_pol2 = calculate_distance_distribution_by_PolII(
    datasets['sim1'],'./sim_outputs'
)

In [ ]:
# plot distance distributions
def plot_distance_distribution(all_data, anchors, conditions_to_plot, savepath=None):
    fig, ax = plt.subplots(figsize=(1.5,1.5)) 
        
    ax.grid(False)
    dist_dict = all_data[anchors]
    max_bin = int(np.max([np.quantile(dist_dict[c],0.99) for c in dist_dict.keys()]))
    bins = np.linspace(0, max_bin, 75) 
    lines = []
    labels = []

    for cond, color in conditions_to_plot.items():
        data = dist_dict[cond]
        if len(data) == 0:
            continue
            
        mean_val = data.mean()
        label_name = name_map[cond]
        
        counts, bin_edges = np.histogram(data, bins=bins, density=True)
        
        counts_scaled = counts * 1e3 
        
        ax.stairs(counts_scaled, bin_edges, color=color, linewidth=1.0,alpha=1)
        
        lines.append(plt.Line2D([0], [0], color=color, linewidth=1.0))
        labels.append(f"{label_name}\nmean = {mean_val:.0f} nm")

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_color('black')
    ax.spines['left'].set_linewidth(0.5)

    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_color('black')
    ax.spines['bottom'].set_linewidth(0.5)

    ax.set_xlim(left=0)
    ax.set_ylim(top=3)
    # ax.set_xticks([0, 250, 500, 750, 1000, 1250])

    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True, nbins=5))

    ax.text(0.02, 1.0, r"x $10^{-3}$", transform=ax.transAxes, fontsize=6, color='black', va='bottom', ha='left')

    ax.set_xlabel(r"3D distance, $\mathit{R}$ [nm]", fontsize=6, color='black')
    ax.set_ylabel(r"Probability density [nm$^{-1}$]", fontsize=6, color='black')

    # ax.set_title("3D distance distributions (PDF)", fontsize=7, color='black', pad=15)

    ax.tick_params(axis='both', which='major', labelsize=6, labelcolor='black', 
                    color='black', direction='out', width=0.5, length=4, 
                    bottom=True, left=True)

    leg = ax.legend(lines, labels, frameon=False, fontsize=6, loc='upper right', 
                    handlelength=1.5, labelspacing=0.8)

    plt.tight_layout()
    if savepath!=None:
        plt.savefig(savepath,dpi=300,transparent=True,bbox_inches='tight')
conds_to_plot = {'cond1':'k'} # change as needed
plot_distance_distribution(all_dists,'sim1',conds_to_plot,
                           savepath='./distance_distributions.svg')

In [ ]:
def plot_msd(msd_data,dist_data,nl_data,anchors,conds_to_plot,savepath=None):
    msd_dict = msd_data[anchors]
    dist_dict = dist_data[anchors]
    nl_dataset = nl_data[anchors]
    fig,ax=plt.subplots(1,1,figsize=(1.3,1.4))
    for c, fmt in conds_to_plot.items():
            rmsd = np.nanmean((dist_dict[c]/1000)**2)
            ax.plot(np.arange(len(msd_dict[c]))*nl_dataset[c][0].meta['Δt']*0.001111111,msd_dict[c]*0.0151**2,fmt,
                    label=name_map[c]+f"\n2<$\\it{{R}}^2$>={2*rmsd:.2f}\u03bcm$^2$",
                    alpha=0.8)
            ax.plot(np.arange(len(msd_dict[c]))*nl_dataset[c][0].meta['Δt']*0.001111111,
                    [2*rmsd]*len(msd_dict[c]),color=fmt,linestyle=':',alpha=0.8)
    plt.xscale('log')
    plt.yscale('log')
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_color('black')
    ax.spines['left'].set_linewidth(0.5)

    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_color('black')
    ax.spines['bottom'].set_linewidth(0.5)
    ax.yaxis.set_major_locator(ticker.LogLocator(10,subs='all'))
    ax.tick_params(axis='both', which='major', labelsize=6, labelcolor='black', 
                    color='black', direction='out', width=0.5, length=4, 
                    bottom=True, left=True)
    ax.legend()
    ax.set_xlim(left=10**2)
    ax.set_ylim(top=1.25)
#     ax.set_ylim(bottom=0.95*np.min([msd_dict[c][10**2] for c in conds_to_plot.keys()]))
    ax.set_xlabel('Time lag [s]')
    ax.set_ylabel('MSD [\u03bcm$^2$]')
    if savepath!=None:
        plt.savefig(savepath,dpi=300,
                transparent=True,bbox_inches='tight')
conds_to_plot = {'cond1':'k'}
plot_msd(MSDs,all_dists,datasets,'sim1',conds_to_plot,savepath='./MSDs.svg')

In [ ]:
# plot distance CDFs stratified by RNAPII
fig,ax = plt.subplots(figsize=(2,1.5))
sns.ecdfplot(refined_dists_by_pol2['cond1']['ON'],ax=ax,label='ON',
             color='#009988',alpha=0.8)
sns.ecdfplot(refined_dists_by_pol2['cond1']['OFF'],ax=ax,label='OFF',
             color='#bbbbbb',alpha=0.8)
ax.set_xlabel('3D distance [nm]')
ax.set_ylabel('Cumulative probability')
ax.set_xlim(left=0)
ax.legend()
plt.savefig('dist_cdf_by_pol2.svg',
            transparent=True,bbox_inches='tight')
plt.show()